# Class 3: Bivariate & Multivariate Analysis and Feature Scaling

### Topics Covered:
1. **Bivariate Analysis**:
   - Numerical vs Numerical (Scatter plots, Correlation matrix, Pearson & Spearman correlation)
   - Categorical vs Numerical (Box plots, Violin plots, Bar plots)
   - Categorical vs Categorical (Cross-tabulation, Count plots, Heatmaps)
2. **Multivariate Analysis**:
   - Pairplots
   - Correlation Heatmaps
   - Multi-feature interactions with `hue` and facet grids
3. **Feature Scaling**:
   - Why feature scaling is essential for distance-based ML algorithms
   - **Standardization (`StandardScaler`)**: transforms features to $\mu=0, \sigma=1$
   - **Normalization / Min-Max Scaling (`MinMaxScaler`)**: bounds features into $[0, 1]$
   - **Robust Scaling (`RobustScaler`)**: uses median and IQR, resistant to outliers
   - Visual comparison of distributions before and after scaling

## 1. Import Libraries & Load Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization theme
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# Load a popular comprehensive dataset (Tips dataset from Seaborn)
df = sns.load_dataset("tips")
print("Dataset Shape:", df.shape)
df.head()

## 2. Bivariate Analysis

Bivariate analysis investigates the relationship between **two** variables.

### A. Numerical vs Numerical: Scatter Plot & Correlation

In [ ]:
# Scatter Plot: Total Bill vs Tip
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="total_bill", y="tip", alpha=0.7, color="royalblue")
plt.title("Bivariate: Total Bill vs Tip")
plt.xlabel("Total Bill ($)")
plt.ylabel("Tip ($)")
plt.show()

# Calculate Pearson and Spearman correlation
numeric_df = df.select_dtypes(include=[np.number])
print("Correlation Matrix:")
numeric_df.corr()

### B. Categorical vs Numerical: Box Plot & Violin Plot

In [ ]:
# Box plot: Total bill distribution across days of the week
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="day", y="total_bill", palette="Set2", ax=axes[0])
axes[0].set_title("Total Bill by Day (Box Plot)")

sns.violinplot(data=df, x="day", y="total_bill", palette="Pastel1", ax=axes[1])
axes[1].set_title("Total Bill by Day (Violin Plot)")

plt.tight_layout()
plt.show()

### C. Categorical vs Categorical: Cross-tabulation & Count Plots

In [ ]:
# Cross-tabulation table between Day and Time (Lunch / Dinner)
cross_tab = pd.crosstab(df["day"], df["time"], margins=True)
print("Cross-tabulation (Day vs Time):")
print(cross_tab)

# Grouped bar plot
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="day", hue="time", palette="coolwarm")
plt.title("Customer Count by Day and Meal Time")
plt.show()

## 3. Multivariate Analysis

Multivariate analysis explores relationships among **three or more** variables simultaneously.

### A. Multi-feature Scatter Plot using `hue` and `style`

In [ ]:
# Total Bill vs Tip by Smoker status and Time
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="total_bill", y="tip", hue="smoker", style="time", s=70)
plt.title("Multivariate: Bill vs Tip by Smoker and Meal Time")
plt.show()

### B. Correlation Heatmap & Pairplot

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Multivariate Correlation Heatmap")
plt.show()

# Pairplot colored by gender
sns.pairplot(df, hue="sex", palette="Set1")
plt.suptitle("Pairplot of Numerical Features by Sex", y=1.02)
plt.show()

## 4. Feature Scaling: Standardization, Normalization, Robust Scaling

### Why Feature Scaling?
Many Machine Learning algorithms (KNN, SVM, Linear Regression with Regularization, PCA, Neural Networks) are sensitive to the magnitude of features.

| Scaler | Formula | Key Properties | Outlier Handling |
|---|---|---|---|
| **StandardScaler** | $z = \frac{x - \mu}{\sigma}$ | Mean = 0, Std = 1 | Sensitive to outliers |
| **MinMaxScaler** | $x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$ | Values in $[0, 1]$ range | Strongly affected by outliers |
| **RobustScaler** | $x_{robust} = \frac{x - Q_2}{Q_3 - Q_1}$ | Uses Median and IQR | **Resilient to outliers** |

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Sample data with varying scales and an outlier
sample_data = pd.DataFrame({
    "Age": [18, 20, 22, 23, 25, 27, 28, 30, 32, 65],  # 65 is an outlier
    "Salary": [25000, 28000, 32000, 35000, 42000, 45000, 50000, 58000, 62000, 250000] # 250000 is an outlier
})

print("Original Data:")
sample_data

In [ ]:
# 1. StandardScaler
std_scaler = StandardScaler()
data_std = pd.DataFrame(std_scaler.fit_transform(sample_data), columns=["Age_Std", "Salary_Std"])

# 2. MinMaxScaler
minmax_scaler = MinMaxScaler()
data_minmax = pd.DataFrame(minmax_scaler.fit_transform(sample_data), columns=["Age_MinMax", "Salary_MinMax"])

# 3. RobustScaler
robust_scaler = RobustScaler()
data_robust = pd.DataFrame(robust_scaler.fit_transform(sample_data), columns=["Age_Robust", "Salary_Robust"])

# Comparison DataFrame
comparison_df = pd.concat([sample_data, data_std, data_minmax, data_robust], axis=1)
comparison_df

### Visual Comparison: Effect of Scalers on Distribution

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

sns.kdeplot(sample_data["Salary"], fill=True, color="black", ax=axes[0])
axes[0].set_title("Original Salary")

sns.kdeplot(data_std["Salary_Std"], fill=True, color="blue", ax=axes[1])
axes[1].set_title("StandardScaler (mean=0, std=1)")

sns.kdeplot(data_minmax["Salary_MinMax"], fill=True, color="green", ax=axes[2])
axes[2].set_title("MinMaxScaler ([0, 1])")

sns.kdeplot(data_robust["Salary_Robust"], fill=True, color="purple", ax=axes[3])
axes[3].set_title("RobustScaler (IQR Robust)")

plt.tight_layout()
plt.show()